# NestedSimPy — Carwash

[NestedSimPy](https://nestedsimpy.github.io/) adapts SimPy's official [Carwash example](https://simpy.readthedocs.io/en/latest/examples/carwash.html) — cars sharing a bank of washing machines (a `Resource`) — so the outer behavior is unchanged while inner simulations fork at each triggering event. This example uses `NestedResource`.

See the [example page](https://nestedsimpy.github.io/official-parity/carwash.html) for the side-by-side plain/nested code.

## 1. Install

_Pre-release: NestedSimPy installs from a hosted wheel. (After the public release this becomes `pip install nestedsimpy`.)_

In [ ]:
# Pre-release install from a hosted wheel (Google Drive).
!pip install -q gdown
import gdown
gdown.download(id="1N7mlgDVpVids6Ekr4p2e-gEUrUodiEuq",
               output="nestedsimpy-0.1.0-py3-none-any.whl", quiet=True)
!pip install -q "nestedsimpy-0.1.0-py3-none-any.whl[plot]"

import nestedsimpy
print("NestedSimPy ready —", len(nestedsimpy.__all__), "public objects")

## 2. Run the nested example

The model is written to a file and run as a subprocess; the output below is the **outer** trajectory and matches the plain SimPy example.

In [ ]:
%%writefile carwash_colab.py
# --- inline prelude (replaces the examples' local _imports shim) ---
import argparse, os, random, shutil, sys, itertools
from pathlib import Path

import simpy
import nestedsimpy
from nestedsimpy import (
    NestedEnvironment, NestedResource, NestedPreemptiveResource,
    NestedStore, NestedContainer,
)
try:
    from nestedsimpy.postprocess import (
        package_latest_run, relocate_raw_artifacts, export_realizations,
    )
except Exception:  # pragma: no cover
    package_latest_run = relocate_raw_artifacts = export_realizations = None

DEFAULT_OUT_ROOT = Path("nested_output")
DEFAULT_AUTOPLOT = False
REPO_ROOT = Path(".")
PACKAGE_ROOT = Path(".")

def default_out(*parts):
    p = DEFAULT_OUT_ROOT.joinpath(*map(str, parts)); p.mkdir(parents=True, exist_ok=True); return p

def set_nested_output_folder(*parts):
    p = Path(os.path.join(*[str(x) for x in parts])); p.mkdir(parents=True, exist_ok=True); return p

def should_emit_user_output(env=None):
    if env is None:
        return True
    f = getattr(env, "should_emit_user_output", None)
    return bool(f()) if callable(f) else getattr(env, "_ns_run_kind", "outer") != "inner"

def user_print(*args, env=None, **kw):
    if should_emit_user_output(env):
        print(*args, **kw)
# --- end prelude ---

"""
Carwash example (nested-sim adaptation).

Covers:

- Waiting for other processes
- Resources: Resource

Scenario:
  A carwash has a limited number of washing machines and defines
  a washing processes that takes some (random) time.

  Car processes arrive at the carwash at a random time. If one washing
  machine is available, they start the washing process and wait for it
  to finish. If not, they wait until they can use one.
"""

import itertools


# fmt: off
RANDOM_SEED = 42
NUM_MACHINES = 2  # Number of machines in the carwash
WASHTIME = 5      # Minutes it takes to clean a car
T_INTER = 7       # Create a car every ~7 minutes
SIM_TIME = 20     # Simulation time in minutes
# fmt: on

# Base output root; nested_run post-processing will create
# run_id/raw and run_id/exports.
NESTED_OUTPUT_FOLDER = set_nested_output_folder("simpy_examples", "carwash")


class Carwash:
    """A carwash has a limited number of machines (``NUM_MACHINES``) to
    clean cars in parallel.

    Cars have to request one of the machines. When they got one, they
    can start the washing processes and wait for it to finish (which
    takes ``washtime`` minutes).

    """

    def __init__(self, env, num_machines, washtime):
        self.env = env
        # Register so nested env can find it by nested_id during branching.
        self.machine = env.register(NestedResource(env, num_machines, nested_id="wash"))
        self.washtime = washtime

    def wash(self, car):
        """The washing processes. It takes a ``car`` processes and tries
        to clean it."""
        yield self.env.nested_timeout(
            {"distribution": "deterministic", "value": self.washtime}, label="wash_time"
        )
        pct_dirt = random.randint(50, 99)
        user_print(f"Carwash removed {pct_dirt}% of {car}'s dirt.", env=self.env)


def car(env, name, cw):
    """The car process (each car has a ``name``) arrives at the carwash
    (``cw``) and requests a cleaning machine.

    It then starts the washing process, waits for it to finish and
    leaves to never come back ...

    """
    user_print(f"{name} arrives at the carwash at {env.now:.2f}.", env=env)
    with cw.machine.request() as request:
        yield request

        user_print(f"{name} enters the carwash at {env.now:.2f}.", env=env)
        yield env.process(cw.wash(name))

        user_print(f"{name} leaves the carwash at {env.now:.2f}.", env=env)


def setup(env, carwash, t_inter):
    """Create initial cars and keep creating cars approx. every ``t_inter`` minutes."""
    car_count = itertools.count()

    # Create 4 initial cars
    for _ in range(4):
        env.process(car(env, f"Car {next(car_count)}", carwash))

    # Create more cars while the simulation is running
    while True:
        delay = random.randint(t_inter - 2, t_inter + 2)
        yield env.nested_timeout(
            {"distribution": "deterministic", "value": delay}, label="interarrival"
        )
        env.process(car(env, f"Car {next(car_count)}", carwash))


def main():
    """Run the nested-sim variant and package the outputs."""

    random.seed(RANDOM_SEED)
    env = NestedEnvironment()
    env._ns_print_branch_summary = False
    carwash = Carwash(env, NUM_MACHINES, WASHTIME)
    # Output options first.
    env.set_output_options(out_dir=str(NESTED_OUTPUT_FOLDER), gzip_trace=False)
    env.set_post_processing_options(
        gzip_trace=False,
        package_latest=True,
        print_outputs=False,
        autoplot=DEFAULT_AUTOPLOT,
        autoplot_example="carwash",
    )
    # Outer settings.
    env.set_rng("independent")
    env.set_outer_seed(RANDOM_SEED)
    env.set_nested_triggering_objects(nested_id="wash")
    env.set_nesting_conditions({"on": "arrival", "frequency": 1})
    env.set_outer_stopping_condition(timeout=SIM_TIME)
    # Inner settings.
    env.set_inner_repetitions(2)
    env.set_inner_stopping_condition(
        relative_time=20.0, triggering_customer_departs=True
    )

    env.process(setup(env, carwash, T_INTER))

    env.nested_run()


if __name__ == "__main__":
    main()


In [ ]:
# Run as a subprocess so the outer output is clean (inner branches fork).
!python carwash_colab.py

## 3. Inspect the run

`OutputManager` reads the run folder and reports the triggering events, plots the outer trajectory, and exports the sample path.

In [ ]:
import glob, os
run = os.path.dirname(glob.glob("simpy_examples/carwash/**/raw", recursive=True)[0])

from nestedsimpy import OutputManager
om = OutputManager(run)
print(f"{len(om.triggers())} triggering events; the outer path has {len(om.export_outer())} recorded events")

fig = om.visualize_outer()        # outer trajectory with triggering markers
fig.show()

om.export_outer("outer.csv")      # the outer sample path as a CSV
print("wrote outer.csv")